# レッスン 13 - エージェントのメモリ


## セットアップ

このノートブックでは、**Microsoft Agent Framework**（MAF）を使って<strong>永続メモリ</strong>を備えた旅行予約エージェントを構築する方法を示します。

どのタイプのエージェントメモリ（作業用、短期、長期）が、会話を超えて情報を保持・利用する方法にどのように影響するかを学びます。

**前提条件:**
- 展開されたチャットモデル（例: `gpt-4.1-mini`）を持つ Microsoft Foundry プロジェクト。
- Azure CLI にログイン済み — ターミナルで `az login` を実行してください。
- `AZURE_AI_PROJECT_ENDPOINT` — あなたの Microsoft Foundry プロジェクトのエンドポイント。
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 展開したモデルの名前。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## エージェントメモリの種類

AI エージェントは、それぞれ異なる目的を持つさまざまな種類のメモリを活用できます：

### ワーキングメモリ
会話スレッド自体—単一セッション内で交わされるメッセージ。エージェントは同じスレッド内の以前のメッセージを参照して一貫性を維持できます。MAF ではこれは **`agent.create_session()`** を通じて作成され、`AgentSession` が返されます。

### 短期メモリ
タスクまたはセッションの期間中保持される情報で、永久的には保存されません。たとえば、エージェントは複数回の計画会話の間に事実を蓄積し、それを使って最終的な旅程を作成することがあります。

### 長期メモリ
<strong>セッション間</strong>で保持される好みや事実。戻ってきたユーザーが食事制限や旅行スタイルを繰り返す必要がないようにします。長期メモリは通常、外部のストア（データベース、ファイル、ベクターインデックスなど）によってサポートされており、ツールを介してエージェントに提供されます。


## セッションを使った作業メモリ

メモリの最も単純な形は会話セッションです。同じセッション（`agent.create_session()`で作成）を連続した`agent.run()`呼び出しに渡すと、エージェントはその会話の全履歴を見て、以前の詳細を思い出すことができます。

旅行代理店を作り、作業メモリを実演しましょう。


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

エージェントは両方のメッセージが同じセッションを共有しているため、予算を正しく記憶していました。これは<strong>作業記憶</strong>です — セッションの存続期間のみ存在します。

### 新しいスレッドでは何が起こるのか？

<strong>新しい</strong>セッションを作成すると、エージェントは前の会話の記憶を持ちません：


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 長期記憶パターン

ユーザーの好みを<strong>セッションを超えて</strong>記憶するには、会話スレッドの外に存在する永続的なストアが必要です。エージェントはこのストアに<strong>ツール</strong>を通じてアクセスします。ツールとは、情報を保存および取得するために呼び出せる関数です。

以下では、シンプルなインメモリの好みストアを実装します（本番環境ではデータベースやベクターインデックスでバックアップします）そしてエージェントが使用できるツールとして公開します。

### アーキテクチャ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### シナリオ 1 — 初めてのユーザーが記念日の旅行を予約する

サラは初めて訪れます。エージェントはツールを使って彼女の好みを保存し、それをホテルの推薦に活用するべきです。


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### シナリオ 2 — サラが数週間後に戻ってくる

サラは<strong>まったく新しいスレッド</strong>を開始します（新しいセッションをシミュレートしています）。作業メモリは空ですが、長期の好みのストアには彼女の情報がまだ残っています。エージェントはそれを取得して、パーソナライズされた推薦に使用する必要があります。


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## まとめ

このレッスンでは、3種類のエージェントメモリと、それをMicrosoft Agent Frameworkで実装する方法を学びました：

| メモリの種類 | MAFメカニズム | 寿命 |
|---|---|---|
| **作業用（Working）** | `agent.create_session()` | 単一の会話 |
| **短期（Short-term）** | スレッド内で蓄積されたコンテキスト | 単一タスク／セッション |
| **長期（Long-term）** | `@tool`関数経由でアクセスされる外部ストア | セッション間 |

### 重要なポイント
1. **`agent.create_session()`** は作業用メモリを提供します — エージェントはセッション内の全ての会話履歴を把握します。
2. <strong>新しいセッションではコンテキストが失われる</strong> — 長期メモリがなければエージェントは過去の会話を思い出せません。
3. **`@tool`関数が橋渡しをします** — エージェントが持続的なストアに情報を保存・取得できるようにします。
4. <strong>パーソナライズは時間と共に向上します</strong> — 好みが多く保存されるほどエージェントの提案も良くなります。

### 実世界の応用例
- <strong>カスタマーサービス</strong>：顧客の履歴と好みを記憶
- <strong>パーソナルアシスタント</strong>：数日または数週間にわたるコンテキスト維持
- <strong>ヘルスケア</strong>：患者情報と好みの追跡
- **Eコマース**：履歴に基づくパーソナライズドショッピング

### 次のステップ
- メモリ用辞書をデータベースやベクターストア（例：Azure AI Search）に置き換える
- 時間に敏感な情報のためにメモリの有効期限を追加する
- 共有メモリを使ったマルチエージェントシステムを構築する
- 知識グラフで支援されたメモリのCogneeノートブックを探求する


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
